# Prototype: droiddetect

Reproduction check for `detectors/droiddetect.py`. Not run by the pipeline: upload to Colab (File > Upload notebook).

**Runtime:** T4 GPU.

**Decides:** Settle DroidDetect Large vs Base on the DroidCollection test split: weighted F1 vs Table 6 (99.25 / 99.18, tolerance 1.0), the pooling that reproduces it, FPR and miss rate on the Python algorithmic slice, ms per sample.

Run top to bottom, then paste the printed comparison back into the chat.

In [ ]:
# %% [1] Setup  (Colab, T4 runtime)
!pip -q install "transformers>=4.48" scikit-learn   # ModernBERT needs >=4.48; the training repo pins nothing
import time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, transformers
from huggingface_hub import hf_hub_download
from transformers import AutoConfig, AutoModel, AutoTokenizer
from sklearn.metrics import f1_score, roc_auc_score
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))

MODELS = {  # repo, pinned revision, backbone, paper 2-class weighted F1 (Table 6 "DroidDetect" row = Table 10 @512)
    "large": ("project-droid/DroidDetect-Large-Binary", "44c45cbfb2dbf307e66acc691440f0cab3a0ddf5", "answerdotai/ModernBERT-large", 99.25),
    "base":  ("project-droid/DroidDetect-Base-Binary",  "4328ba9443dd870cf7b645571ab9a96e5eed4c2a", "answerdotai/ModernBERT-base",  99.18),
}
DATA_REV = "9a42843be99456a3af70d2b9c9a53a4d7624598f"
TOL, MAX_LEN, P_AI, SEED = 1.0, 512, 1, 0          # P_AI: index 1 = MACHINE_GENERATED
N_PAPER, N_PYALGO, N_EXTRA = 12_000, 3_000, 1_500
PAPER_LANGS = {"C", "C++", "C#", "Go", "Java", "Python", "JavaScript"}  # paper: 6 groups, C/C++ merged
DOMAIN = {**dict.fromkeys(["THEVAULT_FUNCTION", "THEVAULT_INLINE", "THEVAULT_CLASS", "STARCODER_DATA", "DROID_PERSONAHUB"], "General"),
          **dict.fromkeys(["TACO", "LEETCODE", "CODEFORCES", "ATCODER"], "Algorithmic"),
          **dict.fromkeys(["OBSCURA_CODER", "ARXIV"], "Research/DS")}
BIN = ["HUMAN_GENERATED", "MACHINE_GENERATED"]     # binary training LABEL_MAPPING; refined/adversarial excluded

In [ ]:
# %% [2] Held-out data: DroidCollection test split (105,826 rows)
df = pd.read_parquet(hf_hub_download("project-droid/DroidCollection", "data/test-00000-of-00001.parquet",
                                     repo_type="dataset", revision=DATA_REV))
df["domain"] = df.Source.map(DOMAIN)
assert df.domain.notna().all(), df.Source[df.domain.isna()].unique()
is_bin, in_lang = df.Label.isin(BIN), df.Language.isin(PAPER_LANGS)
pool = df[is_bin & in_lang]
pyalgo = pool[(pool.Language == "Python") & (pool.domain == "Algorithmic")]
idx = {
    "paper":    pool.sample(N_PAPER, random_state=SEED).index,           # uniform => unbiased pooled F1
    "pyalgo":   pyalgo.sample(min(N_PYALGO, len(pyalgo)), random_state=SEED).index,
    "research": pool[pool.domain == "Research/DS"].index,                 # all of it, ~tiny
    "php_rust": df[is_bin & ~in_lang].index,
    "refined":  df[df.Label == "MACHINE_REFINED"].sample(N_EXTRA, random_state=SEED).index,
    "adv":      df[df.Label == "MACHINE_GENERATED_ADVERSARIAL"].sample(N_EXTRA, random_state=SEED).index,
}
ev_idx = idx["paper"]
for k in list(idx)[1:]:
    ev_idx = ev_idx.union(idx[k])
ev = df.loc[ev_idx].copy()
ev["y"] = (ev.Label != "HUMAN_GENERATED").astype(int)
print(len(ev), "rows to score;", ev.Label.value_counts().to_dict())

tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-large")  # identical vocab/merges to the DroidDetect repos
enc = tok(list(ev.Code), max_length=MAX_LEN, padding="max_length", truncation=True, return_tensors="np")
ids, am = enc["input_ids"], enc["attention_mask"]
ev["ntok"] = am.sum(1)

In [ ]:
# %% [3] Model: minimal TLModel (card / src/train_with_tl.py) without the triplet-loss duplicate encoder
class Droid(nn.Module):
    def __init__(self, backbone, n_cls, proj):
        super().__init__()
        cfg = AutoConfig.from_pretrained(backbone)
        cfg.reference_compile = False
        self.text_encoder = AutoModel.from_config(cfg, attn_implementation="sdpa")  # T4 has no FlashAttention-2
        self.text_projection = nn.Linear(cfg.hidden_size, proj)
        self.classifier = nn.Linear(proj, n_cls)

def load(repo, rev, backbone):
    sd = torch.load(hf_hub_download(repo, "pytorch_model.bin", revision=rev), map_location="cpu", weights_only=True)
    sd = {k: v for k, v in sd.items() if not k.startswith("additional_loss.")}
    m = Droid(backbone, n_cls=sd["classifier.weight"].shape[0], proj=sd["text_projection.weight"].shape[0])
    m.load_state_dict(sd, strict=True)
    print(repo, "hidden", m.text_projection.in_features, "proj", m.text_projection.out_features)
    return m.cuda().eval()

POOLS = {  # card code = .mean(dim=1) over all 512 positions; what the pad positions hold depends on the attention backend
    "A_card_mean512": lambda h, mk: h.mean(1),                     # sdpa: pad positions hold real hidden states
    "B_fa2_sum/512":  lambda h, mk: (h * mk).sum(1) / h.shape[1],  # FA2 (ModernBERT's default when installed) zero-fills pads
    "C_masked_mean":  lambda h, mk: (h * mk).sum(1) / mk.sum(1),   # diagnostic only, not how it was trained
}

@torch.no_grad()
def run(m, ids, am, bs):
    out = {k: [] for k in POOLS}
    for i in range(0, len(ids), bs):
        x, a = torch.from_numpy(ids[i:i+bs]).cuda(), torch.from_numpy(am[i:i+bs]).cuda()
        with torch.autocast("cuda", dtype=torch.float16):     # training used fp16=True
            h = m.text_encoder(input_ids=x, attention_mask=a).last_hidden_state
        h, mk = h.float(), a.unsqueeze(-1).float()
        for k, pool_fn in POOLS.items():
            out[k].append(m.classifier(F.relu(m.text_projection(pool_fn(h, mk)))).softmax(-1)[:, P_AI].cpu())
    return {k: torch.cat(v).numpy() for k, v in out.items()}

def timing(m, n=256):
    res = {}
    for bs in (8, 16, 32, 64):
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        run(m, ids[:bs], am[:bs], bs)                         # warm-up
        torch.cuda.synchronize(); t = time.perf_counter()
        run(m, ids[:n], am[:n], bs)
        torch.cuda.synchronize()
        res[bs] = ((time.perf_counter() - t) / n, torch.cuda.max_memory_allocated() / 2**30)
        print(f"  bs={bs:3d}  {1000*res[bs][0]:6.1f} ms/sample  peak {res[bs][1]:.1f} GiB")
    return res

In [ ]:
# %% [4] Score both models
scores, speed = {}, {}
for name, (repo, rev, backbone, _) in MODELS.items():
    m = load(repo, rev, backbone)
    speed[name] = timing(m)
    best_bs = min(speed[name], key=lambda b: speed[name][b][0])
    t = time.perf_counter(); scores[name] = run(m, ids, am, best_bs)
    print(f"{name}: {len(ids)} rows in {time.perf_counter()-t:.0f}s at bs={best_bs}")
    del m; torch.cuda.empty_cache()

In [ ]:
# %% [5] Compare
def mask(name):
    return ev.index.isin(idx[name])

def stats(p, sel):
    y, pr = ev.y.values[sel], (p[sel] > 0.5).astype(int)
    hum, ai = y == 0, y == 1
    return dict(n=int(sel.sum()), f1=100 * f1_score(y, pr, average="weighted"),
                auroc=100 * roc_auc_score(y, p[sel]) if hum.any() and ai.any() else np.nan,
                fpr=100 * pr[hum].mean() if hum.any() else np.nan, miss=100 * (1 - pr[ai].mean()) if ai.any() else np.nan)

paper, rows = mask("paper"), []
for name, (_, _, _, target) in MODELS.items():
    for pool_name, p in scores[name].items():
        h = stats(p, paper)
        dom = [stats(p, paper & (ev.domain == d).values)["f1"] for d in ("General", "Algorithmic")] + \
              [stats(p, mask("research"))["f1"]]
        pa = stats(p, ((ev.Language == "Python") & (ev.domain == "Algorithmic") & ev.Label.isin(BIN)).values)
        sh = stats(p, paper & (ev.ntok <= 128).values)
        rows.append(dict(model=name, pooling=pool_name, F1=h["f1"], paper=target, ok=abs(h["f1"] - target) <= TOL,
                         AUROC=h["auroc"], FPR=h["fpr"], miss=h["miss"], dom_avg_F1=np.mean(dom),
                         py_algo_F1=pa["f1"], py_algo_FPR=pa["fpr"], py_algo_miss=pa["miss"], py_algo_n=pa["n"],
                         short128_F1=sh["f1"], short_n=sh["n"],
                         refined_recall=100 * (p[mask("refined")] > 0.5).mean(), adv_recall=100 * (p[mask("adv")] > 0.5).mean(),
                         php_rust_F1=stats(p, mask("php_rust"))["f1"],
                         ms_per_sample=1000 * min(v[0] for v in speed[name].values())))
res = pd.DataFrame(rows)
pd.set_option("display.width", 250, "display.max_columns", 30, "display.precision", 2)
print(res.to_string(index=False))

best = res.loc[res.groupby("model")["F1"].idxmax()]
print("\nPooling that reproduces, per model:\n", best[["model", "pooling", "F1", "paper", "ok"]].to_string(index=False))
for name in MODELS:
    pool_name = best.set_index("model").loc[name, "pooling"]
    p = scores[name][pool_name]
    per_lang = {l: round(stats(p, paper & (ev.Language == l).values)["f1"], 2) for l in sorted(PAPER_LANGS)}
    print(name, pool_name, "per-language weighted F1 (cf. Table 4 CLS rows):", per_lang)

In [ ]:
# %% [6] score() for detectors/droiddetect.py = cell 3 (Droid, load, the winning POOLS entry) + this cell
import functools
REPO, REV, BACKBONE = MODELS["large"][:3]
POOLING = best.set_index("model").loc["large", "pooling"]   # in the repo file, hardcode the winning string

@functools.cache
def _load():
    return AutoTokenizer.from_pretrained(BACKBONE), load(REPO, REV, BACKBONE)

@torch.no_grad()
def score(codes, bs=16):
    tok_, m = _load()
    out = []
    for i in range(0, len(codes), bs):
        e = tok_(list(codes[i:i+bs]), max_length=512, padding="max_length", truncation=True, return_tensors="pt").to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            h = m.text_encoder(input_ids=e["input_ids"], attention_mask=e["attention_mask"]).last_hidden_state
        pooled = POOLS[POOLING](h.float(), e["attention_mask"].unsqueeze(-1).float())
        out += m.classifier(F.relu(m.text_projection(pooled))).softmax(-1)[:, 1].tolist()  # P(MACHINE_GENERATED)
    return out

s = score(list(ev.Code.iloc[:64]))
print("score() matches prototype:", np.allclose(s, scores["large"][POOLING][:64], atol=2e-3))